In [0]:
```python
# ============================================================
# STEP 1 — CONFIGURATION AND SOURCE TABLES
# ============================================================

from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.recommendation import ALS

# ============================================================
# PROJECT CONFIGURATION
# Replace placeholders with your environment-specific values.
# ============================================================

PROJECT_ID = "YOUR_GCP_PROJECT_ID"

CURATED_DATASET = "Gcp_Curated"
ML_DATASET = "Gcp_Ml"

# Temporary GCS bucket for the Spark BigQuery connector
BQ_TEMP_BUCKET = "YOUR_BQ_TEMP_BUCKET"

# ============================================================
# CURATED SOURCE TABLES
# ============================================================

FACT_ORDER_TABLE = (
    f"{PROJECT_ID}.{CURATED_DATASET}.FACT_ORDER"
)

DIM_CUSTOMER_TABLE = (
    f"{PROJECT_ID}.{CURATED_DATASET}.DIM_CUSTOMER"
)

MAP_ORDER_ITEM_TABLE = (
    f"{PROJECT_ID}.{CURATED_DATASET}.MAP_ORDER_ITEM"
)

DIM_PRODUCT_TABLE = (
    f"{PROJECT_ID}.{CURATED_DATASET}.DIM_PRODUCT"
)

# ============================================================
# MACHINE LEARNING TABLES
# ============================================================

INTERACTIONS_TABLE = (
    f"{PROJECT_ID}.{ML_DATASET}.CUSTOMER_PRODUCT_INTERACTIONS"
)

RECOMMENDATIONS_TABLE = (
    f"{PROJECT_ID}.{ML_DATASET}.CUSTOMER_PRODUCT_RECOMMENDATIONS"
)

MODEL_METADATA_TABLE = (
    f"{PROJECT_ID}.{ML_DATASET}.CUSTOMER_PRODUCT_ALS_MODEL_METADATA"
)

# ============================================================
# READ SOURCE TABLES FROM BIGQUERY
# ============================================================

fact_orders = (
    spark.read.format("bigquery")
    .option("table", FACT_ORDER_TABLE)
    .load()
)

dim_customer = (
    spark.read.format("bigquery")
    .option("table", DIM_CUSTOMER_TABLE)
    .load()
)

map_order_item = (
    spark.read.format("bigquery")
    .option("table", MAP_ORDER_ITEM_TABLE)
    .load()
)

dim_product = (
    spark.read.format("bigquery")
    .option("table", DIM_PRODUCT_TABLE)
    .load()
)

print("Source tables loaded successfully.")
```

In [0]:

# ============================================================
# STEP 2 — CUSTOMER PRODUCT INTERACTIONS
# ============================================================

fact_orders.createOrReplaceTempView("src_fact_orders")
dim_customer.createOrReplaceTempView("src_dim_customer")
map_order_item.createOrReplaceTempView("src_map_order_item")
dim_product.createOrReplaceTempView("src_dim_product")

interaction_df = spark.sql("""
WITH valid_orders AS (
    SELECT
        order_sk,
        order_id,
        customer_sk,
        order_date,
        order_status
    FROM src_fact_orders
    WHERE order_status IS NOT NULL
      AND UPPER(TRIM(order_status)) NOT IN ('CANCELLED', 'CANCELED')
),

purchase_details AS (
    SELECT
        c.customer_id,
        p.product_id,
        p.product_name,
        p.category,
        p.brand,
        o.order_id,
        o.order_date,
        COALESCE(i.quantity, 0) AS quantity,
        COALESCE(i.line_total, 0) AS line_total
    FROM valid_orders o
    INNER JOIN src_dim_customer c
        ON o.customer_sk = c.customer_sk
       AND c.is_current = TRUE
    INNER JOIN src_map_order_item i
        ON o.order_sk = i.order_sk
    INNER JOIN src_dim_product p
        ON i.product_sk = p.product_sk
    WHERE c.customer_id IS NOT NULL
      AND p.product_id IS NOT NULL
      AND p.is_active = TRUE
),

customer_product_aggregated AS (
    SELECT
        customer_id,
        product_id,
        FIRST(product_name, TRUE) AS product_name,
        FIRST(category, TRUE) AS category,
        FIRST(brand, TRUE) AS brand,
        COUNT(DISTINCT order_id) AS purchase_count,
        SUM(quantity) AS total_quantity,
        SUM(line_total) AS total_spend,
        MIN(order_date) AS first_purchase_date,
        MAX(order_date) AS last_purchase_date
    FROM purchase_details
    GROUP BY customer_id, product_id
)

SELECT
    customer_id,
    product_id,
    product_name,
    category,
    brand,
    purchase_count,
    total_quantity,
    total_spend,
    first_purchase_date,
    last_purchase_date,
    CAST(purchase_count AS DOUBLE) AS interaction_strength,
    CURRENT_TIMESTAMP() AS created_at
FROM customer_product_aggregated
""")

display(interaction_df)


In [0]:

# Save the interaction table in BigQuery

(
    interaction_df.write
    .format("bigquery")
    .option("table", INTERACTIONS_TABLE)
    .option("temporaryGcsBucket", BQ_TEMP_BUCKET)
    .option("writeMethod", "indirect")
    .mode("overwrite")
    .save()
)

print(f"Interaction table saved: {INTERACTIONS_TABLE}")


In [0]:

# ============================================================
# STEP 3 — TRAIN ALS MODEL
# ============================================================

interactions = (
    interaction_df
    .select(
        F.col("customer_id").cast("int").alias("customer_id"),
        F.col("product_id").cast("int").alias("product_id"),
        F.col("interaction_strength").cast("float").alias("rating")
    )
    .filter(
        F.col("customer_id").isNotNull()
        & F.col("product_id").isNotNull()
        & F.col("rating").isNotNull()
        & (F.col("rating") > 0)
    )
    .groupBy("customer_id", "product_id")
    .agg(F.sum("rating").alias("rating"))
)

als = ALS(
    userCol="customer_id",
    itemCol="product_id",
    ratingCol="rating",
    implicitPrefs=True,
    rank=8,
    maxIter=10,
    regParam=0.1,
    coldStartStrategy="drop",
    seed=42
)

als_model = als.fit(interactions)

print("ALS model trained successfully.")


In [0]:

# ============================================================
# STEP 4 — GENERATE TOP 1 RECOMMENDATION PER CUSTOMER
# ============================================================

all_users = interactions.select("customer_id").distinct()
all_products = interactions.select("product_id").distinct()

user_product_pairs = all_users.crossJoin(all_products)

# Use the trained ALS model to score each customer-product pair
predictions = als_model.transform(user_product_pairs)

# Remove products already purchased
purchased_products = interactions.select(
    "customer_id", "product_id"
).distinct()

unseen_predictions = (
    predictions
    .filter(F.col("prediction").isNotNull())
    .join(
        purchased_products,
        ["customer_id", "product_id"],
        "left_anti"
    )
)

# Rank unseen products for each customer
ranking_window = Window.partitionBy("customer_id").orderBy(
    F.desc("prediction"),
    F.asc("product_id")
)

top_recommendations = (
    unseen_predictions
    .withColumn(
        "recommendation_rank",
        F.row_number().over(ranking_window)
    )
    .filter(F.col("recommendation_rank") == 1)
    .withColumnRenamed("prediction", "recommendation_score")
)

# Add active product details
active_products = (
    dim_product
    .filter(F.col("is_active") == True)
    .select("product_id", "product_name", "category", "brand")
    .dropDuplicates(["product_id"])
)

final_recommendations = (
    top_recommendations
    .join(active_products, "product_id", "inner")
    .select(
        "customer_id",
        "product_id",
        "product_name",
        "category",
        "brand",
        "recommendation_score",
        "recommendation_rank"
    )
)

display(final_recommendations)


In [0]:

# ============================================================
# STEP 5 — SAVE VERSIONED ARTIFACTS TO GCS
# ============================================================

run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
run_id = f"run_{run_timestamp}"

MODEL_ROOT = "gs://raw_layer_prac/models/customer_product_als"
RECOMMENDATION_ROOT = (
    "gs://raw_layer_prac/models/customer_product_recommendations"
)

MODEL_PATH = f"{MODEL_ROOT}/{run_id}"
RECOMMENDATION_PATH = f"{RECOMMENDATION_ROOT}/{run_id}"

# Add run information to recommendations
recommendations_for_output = (
    final_recommendations
    .withColumn("model_run_id", F.lit(run_id))
    .withColumn("model_gcs_path", F.lit(MODEL_PATH))
    .withColumn("created_at", F.current_timestamp())
)

# Save the trained Spark ALS model
als_model.write().save(MODEL_PATH)

# Save a historical copy of recommendations for this run
recommendations_for_output.write \
    .mode("error") \
    .parquet(RECOMMENDATION_PATH)

print("Model saved:", MODEL_PATH)
print("Recommendations saved:", RECOMMENDATION_PATH)


In [0]:

# ============================================================
# STEP 6 — WRITE CURRENT RECOMMENDATIONS TO BIGQUERY
# ============================================================

BQ_RECOMMENDATIONS_TABLE = (
    "gcp-data-engineering-504911."
    "Gcp_Ml.CUSTOMER_PRODUCT_RECOMMENDATIONS"
)

BQ_TEMP_BUCKET = "bq"

(
    recommendations_for_output.write
    .format("bigquery")
    .option("table", BQ_RECOMMENDATIONS_TABLE)
    .option("temporaryGcsBucket", BQ_TEMP_BUCKET)
    .option("writeMethod", "indirect")
    .mode("overwrite")
    .save()
)

print("Current recommendations written to:", BQ_RECOMMENDATIONS_TABLE)


In [0]:

# ============================================================
# STEP 7 — SAVE LATEST MODEL METADATA TO BIGQUERY
# ============================================================

from pyspark.sql.types import (
    StructType, StructField, StringType,
    LongType, TimestampType
)

model_count = recommendations_for_output.count()

metadata_schema = StructType([
    StructField("model_run_id", StringType(), False),
    StructField("model_name", StringType(), False),
    StructField("model_type", StringType(), False),
    StructField("model_gcs_path", StringType(), False),
    StructField("recommendation_gcs_path", StringType(), False),
    StructField("recommendation_count", LongType(), False),
    StructField("run_timestamp", TimestampType(), False)
])

metadata_df = spark.createDataFrame(
    [(
        run_id,
        "CUSTOMER_PRODUCT_ALS",
        "Spark MLlib ALS",
        MODEL_PATH,
        RECOMMENDATION_PATH,
        int(model_count),
        datetime.now(timezone.utc).replace(tzinfo=None)
    )],
    schema=metadata_schema
)

BQ_MODEL_METADATA_TABLE = (
    "gcp-data-engineering-504911."
    "Gcp_Ml.CUSTOMER_PRODUCT_ALS_MODEL_METADATA"
)

(
    metadata_df.write
    .format("bigquery")
    .option("table", BQ_MODEL_METADATA_TABLE)
    .option("temporaryGcsBucket", BQ_TEMP_BUCKET)
    .option("writeMethod", "indirect")
    .mode("append")
    .save()
)

print("Latest model metadata saved:", BQ_MODEL_METADATA_TABLE)
print("Run ID:", run_id)
print("Recommendation count:", model_count)
